# שיעור 13 - זיכרון סוכן עם גרפים של ידע Cognee


## התקנה

פנקס זה מדגים כיצד לבנות **עוזר קידוד** חכם עם זיכרון מתמשך באמצעות גרפי ידע של [**Cognee**](https://www.cognee.ai/) ומסגרת הסוכנים של מיקרוסופט (**Microsoft Agent Framework**, MAF).

Cognee הופך טקסט לא מובנה לגרף ידע מובנה שניתן לשאול אותו, הנתמך באמבדינג וקטוריים — מה שנותן לסוכן שלך זיכרון ארוך טווח עשיר ומודע לקשרים.

### מה תלמדו
1. **בניית גרפי ידע**: הפוך פרופילים של מפתחים ושיטות עבודה מומלצות לידע מובנה שניתן לשאול.
2. **שילוב Cognee עם MAF**: השתמש בפונקציות `@tool` כדי לאפשר לסוכן MAF לשאול את גרף הידע של Cognee.
3. **שיחות מודעות למושב**: שמירה על הקשר בין שאלות מרובות באותו מושב.
4. **זיכרון ארוך טווח**: שמור ידע חשוב בין מושבים ושלוף אותו בשיחות חדשות.

### דרישות מקדימות
- פייתון 3.9+
- Redis פועל באופן מקומי (`docker run -d -p 6379:6379 redis`) לניהול מושבים
- מפתח API של LLM (למשל OpenAI) — הגדר `LLM_API_KEY` בקובץ `.env`
- `CACHING=true` בקובץ `.env` (נדרש למושבי Cognee)
- פרויקט Microsoft Foundry עם מודל שיחה מפורסם
- הגדרות `AZURE_AI_PROJECT_ENDPOINT` ו-`AZURE_AI_MODEL_DEPLOYMENT_NAME` בקובץ `.env`
- Azure CLI מחובר (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## סוגי זיכרון של סוכן

מחברת זו חוקרת את שלושת סוגי הזיכרון מאותה מחברת שיעור 13 הראשית, אך משתמשת ב-Cognee כמנוע זיכרון לטווח ארוך:

| סוג זיכרון | מנגנון | אורך חיים |
|---|---|---|
| **עובד** | `agent.create_session()` (MAF) | שיחה יחידה |
| **קצר טווח** | מטמון הפעלות של Cognee (Redis) | הפעלה יחידה |
| **טווח ארוך** | גרף הידע של Cognee + וקטורים | קבוע |

### ארכיטקטורת הזיכרון של Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## הכנת אחסון Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## חלק 1 — בניית בסיס הידע

אנו קולטים שלושה סוגי נתונים ליצירת בסיס ידע מקיף עבור העוזר הקוד שלנו:

1. **פרופיל מפתח** — מומחיות אישית ורקע טכני
2. **שיטות עבודה מומלצות בפייתון** — הזן של פייתון עם קווים מנחים מעשיים
3. **שיחות היסטוריות** — מפגשי שאלות ותשובות מהעבר בין מפתחים לעוזרי בינה מלאכותית


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## הדגמת גרף הידע

Cognee יכול להציג הדמיה אינטראקטיבית בפורמט HTML של הישויות והקשרים שהוא חילץ.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## העשר את הזיכרון עם Memify

`memify()` מנתח את גרף הידע ומייצר חוקים אינטליגנטיים — מזהה תבניות, שיטות עבודה מומלצות ויחסים בין מושגים.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## חלק 2 — סוכן MAF עם כלים של Cognee

עכשיו אנו יוצרים סוכן MAF שיכול לבצע שאילתות לגרף הידע של Cognee דרך פונקציות `@tool`. זה מאפשר לסוכן לנצל את מלוא העוצמה של חיפוש סמנטי מודע גרף תוך שמירת הקשר שיחתי באמצעות מפגשים.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## זיכרון עבודה עם סשנים

ה-`AgentSession` (נוצר באמצעות `agent.create_session()`) מספק זיכרון עבודה בתוך סשן. הסוכן יכול להתייחס להודעות קודמות תוך כדי שאילת ידע בטווח הארוך של גרף הידע של Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## מושב חדש — זיכרון לטווח ארוך נשמר

התחלת מושב חדש מאפסת את זיכרון העבודה, אך גרף הידע של Cognee עדיין זמין. הסוכן יכול לשלוף את אותו ידע לטווח ארוך בשיחה חדשה לגמרי.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## סיכום

במחברת זו בנית עוזר קידוד שמשלב את **זיכרון העבודה של MAF** (`agent.create_session()`) עם **גרף הידע לטווח ארוך של Cognee**.

### מה שלמדת
1. **בניית גרף ידע**: Cognee מקבל טקסט לא מובנה ובונה גרף + זיכרון וקטורי.
2. **העשרת גרף עם memify**: עובדות נגזרות ויחסים עשירים יותר מעל הגרף הקיים שלך.
3. **אינטגרציה בין MAF ל-Cognee**: פונקציות `@tool` מאפשרות לסוכן MAF לשאול את גרף הקוגני באופן טבעי.
4. **זיכרון עבודה + זיכרון לטווח ארוך**: `AgentSession` (דרך `agent.create_session()`) מספק הקשר של המושב בזמן ש-Cognee מספק ידע מתמשך.
5. **חיפוש מסונן עם NodeSets**: מיקוד תתי קבוצה ספציפיות של גרף הידע (למשל רק עקרונות).

### נקודות מפתח
- **Cognee** הופך טקסט גולמי לזיכרון מובנה ומודע ליחסים — חזק יותר מאחסון וקטורי שטוח.
- **פונקציות `@tool`** גשר בין סוכני MAF למערכות ידע חיצוניות באופן נקי.
- **`AgentSession`** (דרך `agent.create_session()`) שומר הקשר לכל שיחה בנפרד מהידע ארוך הטווח.
- אותו גרף ידע משרת מספר מושבים וסוכנים.

### יישומים בעולם האמיתי
- **קופיילוטים למפתחים**: סקירת קוד, ניתוח תקלות, עוזרי ארכיטקטורה
- **קופיילוטים מול לקוחות**: סוכני תמיכה על מסמכי מוצר, שאלות נפוצות, והערות CRM
- **קופיילוטים פנימיים למומחים**: עוזרי מדיניות, משפטיים או אבטחה החושבים לפי הנחיות
- **שכבות נתונים מאוחדות**: שילוב נתונים מובנים ולא מובנים לגרף שניתן לשאול

### צעדים הבאים
- התנסות במודעות תרמית בתוך Cognee
- הגדרת אונטולוגיית OWL לאיכות גרף ספציפית לתחום
- הוספת לולאות משוב משתמש לשיפור אחזור לאורך זמן
- התרחבות למערכות רב-סוכנים המשתפות את שכבת הזיכרון של Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
